In [20]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [5]:
def get_metrics(df: pd.DataFrame, column: str):
    mean_value = df[column].mean()
    std_value = df[column].std()
    # print(f"{column}: {mean_value:.3f} +- {std_value:.3f}")
    return mean_value, std_value

In [18]:
all_metrics = []
folder_path = "../results/modelV5.dim0.5.folds10_id20251126183757"
# folder_path = "../results/modelV2.dim0.5.folds10_id20251120131323"
listed_files = [file for file in os.listdir(folder_path)
                        if (file.endswith(".pkl") and 
                            not file.startswith(".sys.v#."))
                            ]
for file in listed_files:
    data = pd.DataFrame(pd.read_pickle(os.path.join(folder_path, file)))
    data = data[["file_path", "fold", "best_sp_value", "best_fa_value", "best_pd_value"]]
    metrics = {"source_file": None,
                "mean_sp": 0,
                "std_sp": 0,
                "mean_fa": 0,
                "std_fa": 0,
                "mean_pd": 0,
                "std_pd": 0,
           
           }
    # print(f"{file}")
    metrics["mean_sp"], metrics["std_sp"] = get_metrics(data, "best_sp_value")
    metrics["mean_fa"], metrics["std_fa"] = get_metrics(data, "best_fa_value")
    metrics["mean_pd"], metrics["std_pd"] = get_metrics(data, "best_pd_value")
    metrics["source_file"] = file
    # print(f"\n")
    all_metrics.append(metrics)

In [19]:
print(pd.DataFrame(all_metrics).to_latex(index=False))

\begin{tabular}{lrrrrrr}
\toprule
source_file & mean_sp & std_sp & mean_fa & std_fa & mean_pd & std_pd \\
\midrule
iet0.ieta0.pkl & 0.811998 & 0.001928 & 0.117017 & 0.007820 & 0.744018 & 0.007032 \\
iet1.ieta0.pkl & 0.788204 & 0.001837 & 0.131875 & 0.007394 & 0.712178 & 0.007939 \\
iet2.ieta0.pkl & 0.817775 & 0.001406 & 0.120128 & 0.010428 & 0.758003 & 0.009413 \\
iet3.ieta0.pkl & 0.805875 & 0.002944 & 0.132962 & 0.010952 & 0.746990 & 0.007305 \\
iet4.ieta0.pkl & 0.799075 & 0.002776 & 0.143966 & 0.011780 & 0.744131 & 0.008819 \\
iet5.ieta0.pkl & 0.847960 & 0.002398 & 0.114817 & 0.004680 & 0.811541 & 0.003282 \\
iet6.ieta0.pkl & 0.823697 & 0.001756 & 0.132199 & 0.004381 & 0.780753 & 0.004408 \\
iet7.ieta0.pkl & 0.801835 & 0.003006 & 0.135286 & 0.012885 & 0.741411 & 0.012095 \\
\bottomrule
\end{tabular}



In [21]:
import torch.nn as nn
import torch.nn.functional as F
import torch


class ModelV1(nn.Module):
    def __init__(self, input_dim):
        super(ModelV1, self).__init__()
        self.input_dim = input_dim
        self.conv1 = nn.Conv1d(in_channels=1, 
                               out_channels=4, 
                               kernel_size=2, 
                               padding='same') 
        self.conv2 = nn.Conv1d(in_channels=4, 
                               out_channels=8, 
                               kernel_size=2, 
                               padding='same')
        self.fc1_in_features = 8 * input_dim
        self.fc1 = nn.Linear(self.fc1_in_features, 
                             input_dim)
        self.fc2 = nn.Linear(input_dim, 1)

    def forward(self, x):
        x = x.view(-1, 1, self.input_dim) 
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        return x

class ModelV2(nn.Module):
    def __init__(self, input_dim):
        super(ModelV2, self).__init__()
        self.input_dim = input_dim
        self.conv1 = nn.Conv1d(in_channels=1, 
                               out_channels=64, 
                               kernel_size=2, 
                               padding='same') 
        self.conv2 = nn.Conv1d(in_channels=64, 
                               out_channels=32, 
                               kernel_size=2, 
                               padding='same')
        self.fc1_in_features = 32 * input_dim
        self.fc1 = nn.Linear(self.fc1_in_features, 
                             input_dim)
        self.fc2 = nn.Linear(input_dim, 1)

    def forward(self, x):
        x = x.view(-1, 1, self.input_dim) 
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        return x


class ModelV3(nn.Module):
    def __init__(self, input_dim):
        super(ModelV3, self).__init__()
        self.input_dim = input_dim
        self.conv1 = nn.Conv1d(in_channels=1, 
                               out_channels=32, 
                               kernel_size=2, 
                               padding='same' )
        self.conv2 = nn.Conv1d(in_channels=32, 
                               out_channels=16, 
                               kernel_size=2, 
                               padding='same')
        self.fc1 = nn.Linear(in_features=16 * input_dim, 
                             out_features=input_dim)
        self.fc2 = nn.Linear(in_features=input_dim, 
                             out_features=1)

    def forward(self, x):
        x = x.unsqueeze(1) 
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.view(-1, 16 * self.input_dim)
        x = F.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        return x


class ModelV4(nn.Module):
    def __init__(self, input_dim):
        super(ModelV4, self).__init__()
        self.input_dim = input_dim
        self.conv1 = nn.Conv1d(in_channels=1, 
                               out_channels=32, 
                               kernel_size=3, 
                               padding='same' )
        self.conv2 = nn.Conv1d(in_channels=32, 
                               out_channels=16, 
                               kernel_size=3, 
                               padding='same')
        self.fc1 = nn.Linear(in_features=16 * input_dim, 
                             out_features=input_dim)
        self.fc2 = nn.Linear(in_features=input_dim, 
                             out_features=1)

    def forward(self, x):
        x = x.unsqueeze(1) 
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.view(-1, 16 * self.input_dim)
        x = F.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        return x
        

class ModelV5(nn.Module):
    
    def __init__(self, input_dim):
        super(ModelV5, self).__init__()
        self.fc1 = nn.Linear(input_dim, input_dim)
        self.fc2 = nn.Linear(input_dim, 8)
        self.fc3 = nn.Linear(8, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))
        return x

# Modelo v4 deve corrigir esse problema
# /cvmfs/sft.cern.ch/lcg/views/LCG_108_cuda/x86_64-el9-gcc13-opt/lib/python3.12/site-packages/torch/nn/modules/conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /build/jenkins/workspace/lcg_release_pipeline/build/pyexternals/torch-2.7.1/src/torch/2.7.1/aten/src/ATen/native/Convolution.cpp:1036.)
    

def get_model(tag: str, input_dim: int) -> nn.Module:
    if tag == "V1":
        return ModelV1(input_dim)
    if tag == "V2":
        return ModelV2(input_dim)
    if tag == "V3":
        return ModelV3(input_dim)
    if tag == "V4":
        return ModelV4(input_dim)
    if tag == "V5":
        return ModelV5(input_dim)

In [28]:
print(get_model("V5", 50))

ModelV5(
  (fc1): Linear(in_features=50, out_features=50, bias=True)
  (fc2): Linear(in_features=50, out_features=8, bias=True)
  (fc3): Linear(in_features=8, out_features=1, bias=True)
)
